Day 1, Topic 1: Data Type Interpretation (dtype)

## 📘 Topic 1: Data Type Interpretation (`dtype`)

### What is `dtype`?
In NumPy, **every array has a dtype** — a data type that tells NumPy:
- How many **bytes** each element occupies in memory
- How to **interpret** those bytes (integer? float? signed? unsigned?)

| dtype | Bytes | Range |
|-------|-------|-------|
| `int8` | 1 | -128 to 127 |
| `int16` | 2 | -32,768 to 32,767 |
| `int32` | 4 | ~-2.1B to 2.1B |
| `float32` | 4 | ~±3.4 × 10³⁸ |
| `float64` | 8 | ~±1.8 × 10³⁰⁸ |

### 🔑 Key Insight: Bytes Don't Change, Interpretation Does
The **same bytes in memory** can mean completely different numbers depending on the dtype. This is not a copy — it's the same physical memory being read differently.

```
Memory bytes: 41 00 42 00
Read as int16:  [65, 66]      →  ASCII 'A', 'B'
Read as int8:   [65, 0, 66, 0]
Read as float16: [9.14e-5, 9.296e-5]  ← totally different!
```

### `.view()` — Reinterpret Without Copying
`arr.view(new_dtype)` reinterprets the **exact same bytes** under a new dtype. No data is moved or copied.

> ⚠️ **Warning:** The new dtype must be compatible (e.g., total bytes must divide evenly). Otherwise NumPy raises an error.

### `.tobytes()` — See the Raw Bytes
`arr.tobytes()` returns the raw memory as a Python `bytes` object. Append `.hex()` to see it as a readable hex string.

In [1]:
import numpy as np

In [2]:
arr_int8 = np.array([65, 66, 67, 68], dtype=np.int8)

In [3]:
print("Original int 8 array: ", arr_int8)

Original int 8 array:  [65 66 67 68]


In [4]:
print("Memory bytes: ", arr_int8.tobytes())

Memory bytes:  b'ABCD'


In [6]:
arr = np.array([16961, 12345], dtype=np.int16)

In [7]:
print("Array: ", arr)

Array:  [16961 12345]


In [8]:
print("Bytes in hex: ", arr.tobytes().hex())

Bytes in hex:  41423930


In [9]:
view_int8 = arr.view(np.int8)

In [10]:
print("Same bytes viewed as int8: ", view_int8)

Same bytes viewed as int8:  [65 66 57 48]


In [11]:
view_float16 = arr.view(np.float16)

In [12]:
print("Same bytes viewed as float16: ", view_float16)

Same bytes viewed as float16:  [3.127 0.132]


Write a function inspect_bytes(arr) that:

Prints the original array and its dtype.

Prints the raw bytes in hexadecimal.

Then creates a view of the same array as np.int8 and prints that.

Then creates a view as np.float32 (if the number of bytes is divisible by 4) and prints that.

Hint: Use arr.tobytes().hex() to get a clean hex string.

In [20]:
def inspect_bytes(arr):
    print("Original array: ", arr)
    print("Array datatype: ", arr.dtype)
    print("Raw bytes of array in hexadecimal: ", arr.tobytes().hex())
    arr_view_int8 = arr.view(np.int8)
    print("Same bytes viewed as int8: ", arr_view_int8)
    if arr.nbytes % 4 == 0:
        arr_view_float32 = arr.view(np.float32)
        print("Same bytes viewd as float32: ", arr_view_float32)

In [24]:
arr = np.array([126, 325, 68, 45], dtype=np.int16)

In [25]:
arr.dtype

dtype('int16')

## 📘 Topic 2: Memory Layout and Strides (`.strides`, `.flags`)

### What Are Strides?
**Strides** tell NumPy how many **bytes to jump** in memory to move one step along each dimension.

```
arr = np.array([[1, 2, 3, 4],
                [5, 6, 7, 8]], dtype=np.int32)

# int32 = 4 bytes per element
# Row stride: jump 4*4 = 16 bytes to move to the next row
# Col stride: jump 4 bytes to move to the next column
arr.strides → (16, 4)
```

### C-order vs F-order (Row-major vs Column-major)

| | C-order (default) | F-order (Fortran) |
|-|-------------------|-------------------|
| Storage | Row by row | Column by column |
| Fast axis | Last axis (columns) | First axis (rows) |
| Use case | Most Python/NumPy work | MATLAB, LAPACK interop |

```python
arr_c = np.ones((3, 4), dtype=np.int32)           # C-order
arr_c.strides → (16, 4)   # big stride first (rows), small last (cols)

arr_f = np.ones((3, 4), dtype=np.int32, order='F') # F-order  
arr_f.strides → (4, 12)   # small stride first (rows), big last (cols)
```

### ⚡ Performance Implication
Memory is read fastest when accessed **sequentially** (cache-friendly). With a C-order array:
- `np.sum(arr, axis=1)` → sums along columns (sequential in memory) → **FAST**
- `np.sum(arr, axis=0)` → sums along rows (jumping around in memory) → **SLOWER**

### Strides After Slicing
Slicing **changes strides but doesn't copy data**. It just changes the "step size" pointer arithmetic uses.

```python
arr[::2, ::3].strides  # strides are multiplied by the step
```

### Negative Strides = Reversed Arrays
`arr[::-1]` gives **negative strides** — NumPy walks memory backwards. Still a view, no copy!

### `.flags` — Memory Layout Info
```python
arr.flags
# OWNDATA    → True if array owns its memory (not a view)
# C_CONTIGUOUS → True if row-major (C-order) contiguous
# F_CONTIGUOUS → True if column-major (Fortran-order) contiguous
# WRITEABLE   → True if you can modify the array
```

In [26]:
inspect_bytes(arr)

Original array:  [126 325  68  45]
Array datatype:  int16
Raw bytes of array in hexadecimal:  7e00450144002d00
Same bytes viewed as int8:  [126   0  69   1  68   0  45   0]
Same bytes viewd as float32:  [3.618354e-38 4.132693e-39]


Day 1, Topic 2: Memory Layout and Strides (.strides, .flags)

In [47]:
arr = np.array([[1, 2, 3, 4],
                [5, 6, 7, 8],
                [9, 10, 11, 12]], dtype=np.int32)

In [48]:
arr.dtype

dtype('int32')

In [49]:
arr.strides

(16, 4)

In [50]:
arr_f = np.array([[1, 2, 3, 4],
                [5, 6, 7, 8],
                [9, 10, 11, 12]], dtype=np.int32, order='F')

In [51]:
arr_f.strides

(4, 12)

In [52]:
arr_c = np.ones((10000, 10000), dtype=np.float64, order='C')

In [53]:
%timeit np.sum(arr_c, axis=1)

131 ms ± 37.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [54]:
%timeit np.sum(arr_c, axis=0)

74.9 ms ± 5.12 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [55]:
print(arr.flags)

  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False



In [56]:
sliced_arr = arr[:, ::2]

In [57]:
sliced_arr.dtype

dtype('int32')

In [59]:
sliced_arr.strides

(16, 8)

In [60]:
reversed_arr = arr[::-1, ::-1]

In [62]:
print(reversed_arr)

[[12 11 10  9]
 [ 8  7  6  5]
 [ 4  3  2  1]]


In [63]:
print("Strides: ", reversed_arr.strides)

Strides:  (-16, -4)


## 📘 Topic 3: The View vs. Copy Distinction

### Why This Matters
One of the most common sources of bugs in NumPy code is **accidentally modifying data** you didn't mean to, or **accidentally copying** large arrays you didn't need to.

### 🔵 View — Shared Memory
A **view** points to the **same memory** as the original. Modifying the view modifies the original.

```python
original = np.arange(10)
view = original[2:5]   # slice → VIEW
view[0] = 999
# original is now changed too! ← potential bug
```

### 🟢 Copy — Independent Memory
A **copy** allocates **new memory**. Modifying the copy does NOT affect the original.

```python
copy = original[2:5].copy()  # explicit copy
copy[0] = 999
# original is untouched ✓
```

### Quick Reference: View or Copy?

| Operation | Result | Why |
|-----------|--------|-----|
| `arr[2:5]` | **View** | Simple slice = contiguous memory |
| `arr[::2]` | **View** | Stepped slice, uses strides |
| `arr[:, 1:3]` | **View** | 2D slice |
| `arr.reshape(...)` | **View** (usually) | Same data, different shape |
| `arr.T` | **View** | Transpose just swaps strides |
| `arr[[0, 2, 4]]` | **Copy** | Fancy indexing → non-contiguous |
| `arr[arr > 3]` | **Copy** | Boolean indexing |
| `arr.flatten()` | **Copy** | Always copies |
| `arr.ravel()` | **View** (if possible) | Tries to return view first |

### How to Check
```python
# Method 1: .base
view.base is original   # True  → it's a view
copy.base is None       # True  → it's a copy (owns its data)

# Method 2: .flags
view.flags.owndata      # False → doesn't own its memory = view
copy.flags.owndata      # True  → owns its memory = copy
```

### ⚠️ The Inplace vs Out-of-Place Trap
```python
def bad(img_slice):
    img_slice = img_slice * 1.5   # creates NEW array, original untouched
    return img_slice

def also_bad(img_slice):
    img_slice *= 1.5              # INPLACE! if img_slice is a view,
    return img_slice              # this modifies the ORIGINAL array!
```
Use `.copy()` explicitly when you need to work on a region without affecting the original.

Practice: The Stride Predictor
Now, try to predict the strides for these operations without running the code:

A 1D array of 10 float64 elements. What is its stride?

A 2D array of shape (5, 6) with dtype=np.int16 in C-order. What are its strides?

If you slice that 2D array with [::2, ::3], what do the new strides become?

In [64]:
arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

In [65]:
arr.strides

(8,)

In [73]:
arr2d = np.ones((5, 6), dtype=np.int16, order='C')

In [74]:
arr2d.strides

(12, 2)

In [75]:
sliced_arr2d = arr2d[::2, ::3]

In [76]:
sliced_arr2d.strides

(24, 6)

Day 1, Topic 3: The View vs. Copy Distinction

In [78]:
original = np.arange(10)

In [79]:
print("Original: ", original)

Original:  [0 1 2 3 4 5 6 7 8 9]


In [80]:
view = original[2:5]

In [82]:
print("View before change: ", view)

View before change:  [2 3 4]


In [83]:
view[0] = 999
print("View after change: ", view)

View after change:  [999   3   4]


In [84]:
print("Original : ", original)

Original :  [  0   1 999   3   4   5   6   7   8   9]


In [85]:
original = np.arange(10)

In [86]:
print("Original: ", original)

Original:  [0 1 2 3 4 5 6 7 8 9]


In [87]:
view = original[[2, 3, 4]]

In [88]:
view

array([2, 3, 4])

In [89]:
view[0] = 99

In [90]:
view

array([99,  3,  4])

In [91]:
original

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [92]:
x = np.arange(10)

In [93]:
view = x[2:5]

In [94]:
copy = x[[2, 7, 1]]

In [97]:
print("view.base is x:", view.base is x) #True
print("view.flags.owndata:", view.flags.owndata) #False

view.base is x: True
view.flags.owndata: False


In [98]:
print("copy.base is None:", copy.base is None) #True
print("copy.flags.owndata:", copy.flags.owndata) #True

copy.base is None: True
copy.flags.owndata: True


In [105]:
def adjust_brightness(img_slice):
    # Intended to just work on a local copy? Actually, if img_slice is a view, this modifies the original!
    img_slice = img_slice * 1.5
    return img_slice


In [106]:
original_image = np.array([[100, 150], [200, 250]])
region = original_image[0:2, 0:1]   # A view of the first column

In [110]:
adjusted = adjust_brightness(region)

In [111]:
print("Original image:\n", original_image)

Original image:
 [[100 150]
 [200 250]]


In [112]:
def adjust_brightness_inplace(img_slice):
    img_slice *= 1.5
    return img_slice

In [116]:
original_image = np.array([[100, 150], [200, 250]], dtype=np.float64)
region = original_image[0:2, 0:1]

In [117]:
adjusted = adjust_brightness_inplace(region)

In [118]:
print("Original image:\n", original_image)

Original image:
 [[150. 150.]
 [300. 250.]]


In [119]:
safe_copy = original[2:5].copy()

In [120]:
safe_copy[0] = 99

In [121]:
original

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [122]:
x = np.arange(12).reshape(3, 4)

In [123]:
y = x.T

In [124]:
print("y.flags.owndata:", y.flags.owndata)

y.flags.owndata: False


In [125]:
z = y.ravel()

In [126]:
print("z.flags.owndata:", z.flags.owndata)

z.flags.owndata: True


For each operation below, predict whether the result is a view or a copy. Then verify with .base.

In [151]:
arr = np.arange(8).reshape(2, 4)

In [152]:
arr

array([[0, 1, 2, 3],
       [4, 5, 6, 7]])

In [153]:
a = arr[0] #View
print(a.base is arr.base)

True


In [154]:
b = arr[:, 1:3] #view
print(b.base is arr.base)

True


In [157]:
c = arr[arr > 3] #copy
print(c.base is arr.base)

False


In [158]:
d = arr[[0,1]] #copy
print(d.base is arr.base)

False


In [159]:
e = arr.reshape(4,2) #View
print(e.base is arr.base)

True


In [160]:
f = arr.flatten() #copy
print(f.base is arr.base)

False


In [161]:
g = arr.T #view
print(g.base is arr.base)

True


In [162]:
h = arr[0:1, 0:2] #view
print(h.base is arr.base)

True
